# MAPLE restore notebook
Fresh Kaggle session: attach the dataset that holds the `MAPLE-*.tar.gz` artifacts + `checksums.sha256` (and make sure the four GGUF models exist at their original paths). Restores without recompiling llama.cpp and without downloading models.

In [ ]:
import os, sys, json, subprocess, shutil
P = "/kaggle/working/MAPLE-llama.cpp"; ART = "/kaggle/working/artifacts"
SRC_ARCHIVE = "/kaggle/working/downloaded-llama-cpp-full-cuda-t4.tar.gz"   # reusable shared artifact: READ ONLY
os.environ["MAPLE_HOME"] = P
LD = "/usr/local/nvidia/lib64"   # real NVIDIA driver dir (session-only; set per command, never globally)
def sh(cmd, check=False, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, env=dict(os.environ, LD_LIBRARY_PATH=LD + ":" + os.environ.get("LD_LIBRARY_PATH", "")), **kw)
    print((r.stdout + r.stderr).rstrip()); 
    if check and r.returncode: raise SystemExit("FAILED rc=%d: %s" % (r.returncode, cmd))
    return r.returncode

## 1. Locate artifacts

In [ ]:
sh("ls -la /kaggle/input/*/ /kaggle/working/artifacts 2>/dev/null | head -40")

## 2. Restore (checksums, extraction, CUDA compatibility, model paths, smoke test, API health)
The script is the single source of truth. It stops with an exact diagnostic if anything is wrong (exit 2 artifacts, 3 models, 4 CUDA/binaries, 5 tests).

In [ ]:
# If the project is not extracted yet, bootstrap just the restore scripts from MAPLE-complete.tar.gz or MAPLE-runtime.tar.gz
import glob, tarfile
if not os.path.isfile(P + "/scripts/restore_kaggle.py"):
    for cand in glob.glob("/kaggle/input/*/MAPLE-runtime.tar.gz") + glob.glob(ART + "/MAPLE-runtime.tar.gz"):
        with tarfile.open(cand) as t: t.extractall("/kaggle/working"); print("bootstrapped from", cand); break
sh("python3 -I %s/scripts/restore_kaggle.py" % P)

## 3. Start the API and use it

In [ ]:
import time, urllib.request
env = dict(os.environ, LD_LIBRARY_PATH=LD + ":" + P + "/vendor/llamacpp/llama.cpp/build/bin", PYTHONPATH=P + "/src", MAPLE_HOME=P)
api = subprocess.Popen([sys.executable, "-s", "-m", "maple", "serve", "--port", "8000"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
def call(path, body=None):
    r = urllib.request.Request("http://127.0.0.1:8000" + path, json.dumps(body).encode() if body else None, {"Content-Type": "application/json"})
    with urllib.request.urlopen(r, timeout=600) as x: return json.loads(x.read()), dict(x.headers)
for _ in range(60):
    try: print(call("/health")[0]); break
    except Exception: time.sleep(0.5)

In [ ]:
for m in ("deepseek", "gemma", "quant", "fast", "auto"):
    o, h = call("/v1/chat/completions", {"model": m, "messages": [{"role": "user", "content": "Write a git commit message for fixing a null check."}], "max_tokens": 48})
    print(m, "->", h["X-MAPLE-Model"], "|", (o["choices"][0]["message"].get("content") or "")[:80].replace("\n", " "))

In [ ]:
api.terminate()   # stops the API (models unload)